In [1]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict

# NOTE: In this Kaggle image, "keras==3.x" is installed. Using "from keras..." can
# trigger protobuf/tf_keras incompatibilities (e.g., MessageFactory/GetPrototype errors).
# Importing via tf_keras is stable here and keeps the same high-level Keras API.
import tf_keras as keras
from tf_keras.layers import Dense, GlobalAveragePooling1D, Embedding
from tf_keras.callbacks import EarlyStopping
from tf_keras.models import Sequential
from tf_keras.preprocessing.sequence import pad_sequences
from tf_keras.preprocessing.text import Tokenizer
from tf_keras.utils import to_categorical

from sklearn.model_selection import train_test_split

np.random.seed(7)

# Robust Kaggle paths (do not assume ../input)
BASE_INPUT = "/kaggle/input"
DATA_DIR = os.path.join(BASE_INPUT, "spooky-author-identification")
if not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
    # fallback to flat layout if needed
    DATA_DIR = BASE_INPUT

train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")
sample_sub_path = os.path.join(DATA_DIR, "sample_submission.csv")

print("Using paths:")
print("train:", train_path)
print("test :", test_path)
print("sample:", sample_sub_path)



2026-01-21 03:51:40.742787: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768967500.757336      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768967500.761834      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-21 03:51:40.781203: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Using paths:
train: /kaggle/input/spooky-author-identification/train.csv
test : /kaggle/input/spooky-author-identification/test.csv
sample: /kaggle/input/spooky-author-identification/sample_submission.csv


In [2]:
df = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

# pandas>=2 removed .append; use concat (same semantics here)
df_full = pd.concat([df, df_test], axis=0, ignore_index=True, sort=False)

a2c = {"EAP": 0, "HPL": 1, "MWS": 2}
y = np.array([a2c[a] for a in df.author.values])
y = to_categorical(y, num_classes=3)
y.shape



(17621, 3)

In [3]:
counter = {name: defaultdict(int) for name in set(df.author)}
for text, author in zip(df.text, df.author):
    text = text.replace(" ", "")
    for c in text:
        counter[author][c] += 1

chars = set()
for v in counter.values():
    chars |= set(v.keys())

names = [author for author in counter.keys()]

print("c ", end="")
for n in names:
    print(n, end="   ")
print()
for c in chars:
    print(c, end=" ")
    for n in names:
        print(counter[n][c], end=" ")
    print()




c EAP   MWS   HPL   
o 60172 48209 45784 
K 75 30 156 
" 2678 1337 461 
c 21605 16196 16502 
d 32981 31906 29982 
, 15651 10863 7717 
' 1176 426 1573 
X 14 3 5 
: 163 299 44 
S 651 521 757 
ö 12 0 2 
? 457 368 153 
ï 0 0 7 
î 1 0 0 
P 382 314 287 
Y 247 212 94 
ñ 0 0 7 
J 145 60 193 
b 11851 8664 9585 
O 369 245 455 
ë 0 0 10 
u 23540 18966 17463 
Q 20 7 10 
Å 0 0 1 
a 61285 49813 50984 
y 15188 13403 11272 
ç 1 0 0 
r 45833 39732 36412 
Υ 0 0 1 
V 140 51 61 
Π 0 0 1 
g 14459 11366 13426 
M 971 370 580 
Ν 0 0 1 
æ 32 0 10 
R 233 348 209 
Z 15 2 43 
i 54488 41549 39750 
A 1120 847 1041 
l 31655 25074 27162 
W 671 607 649 
E 391 408 247 
Σ 0 0 1 
é 43 0 15 
ü 1 0 5 
k 3814 3338 4746 
I 4339 4464 3140 
F 337 219 236 
h 46016 39427 38489 
f 19981 16537 14538 
D 443 208 300 
ä 1 0 3 
w 15625 14485 14050 
v 8571 7165 5843 
N 368 182 316 
â 6 0 0 
e 102699 88221 79256 
m 20378 18513 15879 
à 8 0 0 
H 791 606 657 
n 56062 45353 45649 
t 73704 57079 55884 
C 372 275 390 
T 1986 1121 1423 
G 280

In [4]:
def preprocess(text):
    text = text.replace("' ", " ' ")
    signs = set(',.:;"?!')
    prods = set(text) & signs
    if not prods:
        return text

    for sign in prods:
        text = text.replace(sign, " {} ".format(sign))
    return text




In [5]:
def create_docs(df, n_gram_max=2):
    def add_ngram(q, n_gram_max):
        ngrams = []
        for n in range(2, n_gram_max + 1):
            for w_index in range(len(q) - n + 1):
                ngrams.append("--".join(q[w_index : w_index + n]))
        return q + ngrams

    docs = []
    for doc in df.text.values:
        doc = preprocess(doc).split()
        docs.append(" ".join(add_ngram(doc, n_gram_max)))

    return docs




In [6]:
min_count = 1

docs = create_docs(df)
docs_full = create_docs(df_full)

tokenizer = Tokenizer(lower=True, filters="")
tokenizer.fit_on_texts(docs_full)

# Keep original logic: compute vocab size via count threshold
num_words = sum([1 for _, v in tokenizer.word_counts.items() if v >= min_count])
num_words



247345

In [7]:
tokenizer = Tokenizer(num_words=num_words, lower=True, filters="")
tokenizer.fit_on_texts(docs_full)

# IMPORTANT: texts_to_sequences must be applied BEFORE pad_sequences (fixes dtype crash)
docs = tokenizer.texts_to_sequences(docs)

print("Samples Number:", len(docs))
print("Sample 1:\n{}\nSample 2:\n{}".format(docs[0], docs[1]))



Samples Number: 17621
Sample 1:
[47, 7, 108, 26, 9431, 2, 1710, 387, 7, 18, 447, 1012, 1842, 12, 2, 14578, 32, 8, 62, 44381, 102, 79, 155, 4, 2374, 480, 208, 75105, 24406, 19962, 44382, 4832, 86, 1711, 75106, 44383, 31649, 115, 44384, 75107, 401, 1521, 75108, 75109, 31650, 12820, 1974]
Sample 2:
[19, 18, 2243, 6, 6832, 69, 445, 20, 28, 46, 1676, 25, 8, 1975, 5, 8582, 1359, 2924, 48, 4, 156, 12821, 9432, 19963, 75110, 19964, 31651, 1038, 14579, 19965, 75111, 888, 10323, 11456, 44385, 75112, 75113, 16836, 488]


In [8]:
max_size = 0
for text in docs:
    max_size = len(text) if len(text) > max_size else max_size
print("Max number of words in a sample for full dataset:", max_size)



Max number of words in a sample for full dataset: 1749


In [9]:
# Keep the same padding length used by the original notebook (1749),
# but ensure maxlen variable is set and reused consistently.
maxlen = 1749
docs = pad_sequences(sequences=docs, maxlen=maxlen)
print(docs.shape)



(17621, 1749)


In [10]:
docs_origin = create_docs(df)
type(docs_origin), docs_origin[0][:200]



(list,
 'So I did not abandon the search until I had become fully satisfied that the thief is a more astute man than myself . So--I I--did did--not not--abandon abandon--the the--search search--until until--I ')

In [11]:
num_words, docs.shape



(247345, (17621, 1749))

In [12]:
# input_dim consistent with original approach: max token id + 1
input_dim = int(np.max(docs)) + 1
embedding_dims = 50
input_dim, embedding_dims



(229877, 50)

In [13]:
x_train, x_test, y_train, y_test = train_test_split(
    docs, y, test_size=0.20, random_state=42
)

x_train.shape, y_train.shape, x_test.shape, y_test.shape




((14096, 1749), (14096, 3), (3525, 1749), (3525, 3))

In [14]:
def create_model(embedding_dims=20, optimizer="adam"):
    model = Sequential()
    model.add(Embedding(input_dim=input_dim, output_dim=embedding_dims))
    model.add(GlobalAveragePooling1D())
    model.add(Dense(3, activation="softmax"))

    model.compile(
        loss="categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"]
    )
    return model




In [15]:
model = create_model(embedding_dims=embedding_dims, optimizer="adam")
model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, None, 50)          11493850  
                                                                 
 global_average_pooling1d (  (None, 50)                0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense (Dense)               (None, 3)                 153       
                                                                 
Total params: 11494003 (43.85 MB)
Trainable params: 11494003 (43.85 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


2026-01-21 03:51:49.998296: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1768967509.998766      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:85:00.0, compute capability: 8.6


In [16]:
epochs = 500
hist = model.fit(
    x_train,
    y_train,
    batch_size=100,
    validation_data=(x_test, y_test),
    epochs=epochs,
    callbacks=[
        EarlyStopping(patience=25, monitor="val_loss", restore_best_weights=True)
    ],
    verbose=2,
)



Epoch 1/500


I0000 00:00:1768967510.614792      65 service.cc:148] XLA service 0x7f79fa4a9340 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768967510.614850      65 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-01-21 03:51:50.621435: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1768967510.635852      65 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1768967510.695956      65 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


141/141 - 12s - loss: 1.0872 - accuracy: 0.4037 - val_loss: 1.0883 - val_accuracy: 0.3969 - 12s/epoch - 88ms/step
Epoch 2/500
141/141 - 9s - loss: 1.0851 - accuracy: 0.4038 - val_loss: 1.0870 - val_accuracy: 0.3969 - 9s/epoch - 62ms/step
Epoch 3/500
141/141 - 9s - loss: 1.0824 - accuracy: 0.4039 - val_loss: 1.0846 - val_accuracy: 0.3969 - 9s/epoch - 62ms/step
Epoch 4/500
141/141 - 8s - loss: 1.0784 - accuracy: 0.4040 - val_loss: 1.0807 - val_accuracy: 0.3969 - 8s/epoch - 54ms/step
Epoch 5/500
141/141 - 6s - loss: 1.0726 - accuracy: 0.4042 - val_loss: 1.0743 - val_accuracy: 0.3972 - 6s/epoch - 45ms/step
Epoch 6/500
141/141 - 6s - loss: 1.0628 - accuracy: 0.4053 - val_loss: 1.0657 - val_accuracy: 0.3986 - 6s/epoch - 40ms/step
Epoch 7/500
141/141 - 5s - loss: 1.0498 - accuracy: 0.4137 - val_loss: 1.0530 - val_accuracy: 0.4051 - 5s/epoch - 35ms/step
Epoch 8/500
141/141 - 5s - loss: 1.0323 - accuracy: 0.4310 - val_loss: 1.0380 - val_accuracy: 0.4332 - 5s/epoch - 35ms/step
Epoch 9/500
141/14

In [17]:
# (Kept as a comment block in original; no-op)
"""
docs = create_docs(df)
tokenizer = Tokenizer(lower=True, filters='')
tokenizer.fit_on_texts(docs)
num_words = sum([1 for _, v in tokenizer.word_counts.items() if v >= min_count])

tokenizer = Tokenizer(num_words=num_words, lower=True, filters='')
tokenizer.fit_on_texts(docs)
docs = tokenizer.texts_to_sequences(docs)

maxlen = 256

docs = pad_sequences(sequences=docs, maxlen=maxlen)

input_dim = np.max(docs) + 1
"""



"\ndocs = create_docs(df)\ntokenizer = Tokenizer(lower=True, filters='')\ntokenizer.fit_on_texts(docs)\nnum_words = sum([1 for _, v in tokenizer.word_counts.items() if v >= min_count])\n\ntokenizer = Tokenizer(num_words=num_words, lower=True, filters='')\ntokenizer.fit_on_texts(docs)\ndocs = tokenizer.texts_to_sequences(docs)\n\nmaxlen = 256\n\ndocs = pad_sequences(sequences=docs, maxlen=maxlen)\n\ninput_dim = np.max(docs) + 1\n"

In [18]:
test_df = pd.read_csv(test_path)
test_docs = create_docs(test_df)
test_docs = tokenizer.texts_to_sequences(test_docs)
test_docs = pad_sequences(sequences=test_docs, maxlen=maxlen)

# Keras 3/tf_keras does not provide predict_proba; predict() returns probabilities for softmax.
y_pred = model.predict(test_docs, batch_size=256, verbose=0)

result = pd.read_csv(sample_sub_path)
# Fill required columns in correct order
for a, i in a2c.items():
    result[a] = y_pred[:, i]

# Ensure id matches test order (sample_submission is already aligned, but be explicit)
result["id"] = test_df["id"].values

result.head()



,id,EAP,HPL,MWS
0,id27251,0.000086,0.999900,0.000014
1,id09612,0.092774,0.030911,0.876316
2,id11943,0.004231,0.003624,0.992145
3,id19526,0.059158,0.107305,0.833537
4,id12931,0.041280,0.006501,0.952219


In [19]:
# Write a valid Kaggle submission with .csv suffix
out_path = "submission.csv"
result.to_csv(out_path, index=False)
print("Wrote:", out_path, "shape:", result.shape)
print(result.columns.tolist())

Wrote: submission.csv shape: (1958, 4)
['id', 'EAP', 'HPL', 'MWS']
